# 004 数组张量与形状思维

先在纸上标注每个数组的轴。固定规则演示，不拟合参数，不作泛化评价。

In [1]:
from pathlib import Path
import numpy as np
from experiment import load_data, array_predict, loop_predict, strict_mae, channel_marker, run
from test_experiment import run_tests
ids, X, y = load_data()
print(X)
print("shape", X.shape, "ndim", X.ndim, "size", X.size, "dtype", X.dtype)

[[ 1. 10.]
 [ 2. 20.]
 [ 3. 30.]]
shape (3, 2) ndim 2 size 6 dtype float64


## 索引保留了哪些轴
相同数字不保证相同shape。

In [2]:
for value in [X[1], X[:, 0], X[:, 0:1]]:
    print(value, "shape", value.shape)
assert X[:, 0].shape == (3,)
assert X[:, 0:1].shape == (3, 1)

[ 2. 20.] shape (2,)
[1. 2. 3.] shape (3,)
[[1.]
 [2.]
 [3.]] shape (3, 1)


## 与循环逐项对照
先乘系数，再沿特征求和，再加偏移。

In [3]:
weights = np.array([2.0, 0.1], dtype=np.float64)
weighted, combined, predictions = array_predict(X, weights, 1.0)
print("weighted", weighted, weighted.shape)
print("combined", combined, combined.shape)
print("predictions", predictions, predictions.shape)
loop = loop_predict(X, weights, 1.0)
assert loop.shape == predictions.shape
np.testing.assert_allclose(loop, predictions, rtol=0, atol=1e-12)

weighted [[2. 1.]
 [4. 2.]
 [6. 3.]] (3, 2)
combined [3. 6. 9.] (3,)
predictions [ 4.  7. 10.] (3,)


## 归约的方向
axis指定被合并的轴；keepdims保留轴位置。

In [4]:
print("axis0", weighted.sum(axis=0))
print("axis1", weighted.sum(axis=1))
print("keepdims", weighted.sum(axis=1, keepdims=True).shape)
print("六项均值", weighted.mean())

axis0 [12.  6.]
axis1 [3. 6. 9.]
keepdims (3, 1)
六项均值 3.0


## 故意制造错误广播
九项配对不是本任务的逐样本误差。

In [5]:
wrong = predictions.reshape(3, 1) - y
print("正确偏差", predictions - y)
print("错误配对", wrong, "shape", wrong.shape)
print("正确MAE", strict_mae(y, predictions))
print("错误两两平均", np.abs(wrong).mean())
try:
    strict_mae(y, predictions.reshape(3, 1))
except ValueError:
    print("形状检查已拒绝列预测与一维标签混用")

正确偏差 [-1.  0.  1.]
错误配对 [[-1. -3. -5.]
 [ 2.  0. -2.]
 [ 5.  3.  1.]] shape (3, 3)
正确MAE 0.6666666666666666
错误两两平均 2.4444444444444446
形状检查已拒绝列预测与一维标签混用


## @与形状重排
reshape保持默认元素顺序，转置交换轴。

In [6]:
W = np.array([[2, -1], [0.1, 0.2]])
print("矩阵乘法", X @ W)
print("两个输出加偏移", X @ W + [1, -1])
print("reshape", X.reshape(2, 3))
print("transpose", X.T)
assert not np.array_equal(X.reshape(2, 3), X.T)

矩阵乘法 [[3. 1.]
 [6. 2.]
 [9. 3.]]
两个输出加偏移 [[ 4.  0.]
 [ 7.  1.]
 [10.  2.]]
reshape [[ 1. 10.  2.]
 [20.  3. 30.]]
transpose [[ 1.  2.  3.]
 [10. 20. 30.]]


## 用唯一编号检查通道
新位置的样本、通道、像素身份保持不变。

In [7]:
nchw, nhwc = channel_marker()
print(nchw.shape, nhwc.shape)
print(nchw[1, 2, 0, 1], nhwc[1, 0, 1, 2])
assert nchw[1, 2, 0, 1] == nhwc[1, 0, 1, 2] == 1201
print(run_tests())
print(run(Path.cwd() / "notebook_outputs"))

(2, 3, 2, 2) (2, 2, 2, 3)
1201 1201
{'status': 'passed', 'test_groups': ['hand-calculated intermediate arrays and loop equivalence', 'reduction axes and keepdims', 'silent pairwise bug reproduced and strict metric rejects invalid inputs', 'feature shape and finite-value checks', 'broadcast compatibility and sample-specific offsets', 'matrix multiplication and reshape versus transpose', 'basic slice view and explicit copy', 'all 24 unique channel markers retain identity']}
{'correct_mae': 0.6666666666666666, 'wrong_pairwise_mean_absolute_error': 2.4444444444444446, 'weighted_axis_0_sum': [12.0, 6.0], 'weighted_axis_1_sum': [3.0, 6.0, 9.0], 'weighted_all_mean': 3.0, 'sample_count': 3}


## 最后重启并全部执行
逐项记录shape、dtype、数值以及轴含义。解释为什么不能通过随意flatten修复不理解的形状错误。